In [4]:
#MARKET BASKET ANALYSIS (Association Rules)
#🎯 Goal:- Find patterns like: “Customers who bought X also bought Y”

In [16]:
#====================
# STEP 1 — LOAD DATA
#====================
import pandas as pd
import sqlite3

conn = sqlite3.connect("retailiq.db")

df = pd.read_sql("""
SELECT InvoiceNo, Description
FROM retail_data
WHERE Quantity > 0
""", conn)

df.head()


#======================
# STEP 2 — CLEAN DATA
#======================
# Remove nulls
df = df.dropna(subset=['Description'])

# Convert to string
df['Description'] = df['Description'].astype(str)

# Remove duplicates
df = df.drop_duplicates()


#==========================================
# STEP 3 — CREATE TRANSACTIONS (OPTIMIZED)
#==========================================
# Keep only top 100 products (performance optimization)

top_products = df['Description'].value_counts().head(100).index
df = df[df['Description'].isin(top_products)]

# Create transaction list
transactions = df.groupby('InvoiceNo')['Description'].apply(list)

transactions.head()


#==========================================
# STEP 4 — ITEM SUPPORT CALCULATION
#==========================================
from collections import Counter

item_counts = Counter()

for items in transactions:
    item_counts.update(set(items))  # unique items per order

total_orders = len(transactions)

item_support = {
    item: count / total_orders
    for item, count in item_counts.items()
}


#==========================================
# STEP 5 — PAIR GENERATION (ASSOCIATIONS)
#==========================================
from itertools import combinations

pair_counts = Counter()

for items in transactions:
    pairs = combinations(set(items), 2)
    pair_counts.update(pairs)


#==========================================
# STEP 6 — CALCULATE SUPPORT, CONFIDENCE, LIFT
#==========================================
results = []

for (item1, item2), count in pair_counts.items():

    support_ab = count / total_orders
    support_a = item_support[item1]
    support_b = item_support[item2]

    confidence = support_ab / support_a
    lift = support_ab / (support_a * support_b)

    results.append([
        item1,
        item2,
        support_ab,
        confidence,
        lift
    ])

rules_df = pd.DataFrame(results, columns=[
    'Item_A',
    'Item_B',
    'Support',
    'Confidence',
    'Lift'
])

rules_df.head()


#==============================
# STEP 7 — FILTER STRONG RULES
#==============================
strong_rules = rules_df[
    (rules_df['Confidence'] > 0.3) &
    (rules_df['Lift'] > 1.5)
]


#==========================
# STEP 8 — SORT BEST RULES
#==========================
strong_rules = strong_rules.sort_values(by='Lift', ascending=False)

strong_rules.head(10)

,Item_A,Item_B,Support,Confidence,Lift
2296,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.024423,0.663669,14.511398
202,SPACEBOY LUNCH BOX,DOLLY GIRL LUNCH BOX,0.024356,0.522727,12.821244
2488,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.023496,0.638489,12.320477
285,PLASTERS IN TIN WOODLAND ANIMALS,PLASTERS IN TIN SPACEBOY,0.017870,0.459966,12.002807
13,ALARM CLOCK BAKELIKE RED,ALARM CLOCK BAKELIKE GREEN,0.035078,0.604333,11.572708
626,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.026011,0.501916,10.974594
6377,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.022503,0.449735,10.700871
799,WOODLAND CHARLOTTE BAG,RED RETROSPOT CHARLOTTE BAG,0.017605,0.500942,10.241850
419,COOK WITH WINE METAL SIGN,HAND OVER THE CHOCOLATE SIGN,0.015885,0.398671,10.209359
7698,LUNCH BAG VINTAGE DOILY,JUMBO BAG VINTAGE DOILY,0.014892,0.418216,10.175233


In [17]:
#====================
# STEP 1 — LOAD DATA
#====================
import pandas as pd
import sqlite3

conn = sqlite3.connect("retailiq.db")

df = pd.read_sql("""
SELECT InvoiceNo, Description, Quantity, CustomerID, InvoiceDate, UnitPrice
FROM retail_data
WHERE Quantity > 0
""", conn)

df.head()


#======================
# STEP 2 — CLEAN DATA
#======================
# Remove nulls
df = df.dropna(subset=['Description', 'CustomerID'])

# Convert types
df['Description'] = df['Description'].astype(str)
df['CustomerID'] = df['CustomerID'].astype(int)

# Remove duplicates
df = df.drop_duplicates()


#==========================================
# STEP 3 — CREATE TRANSACTIONS (OPTIMIZED)
#==========================================
# Keep only top 100 products (performance optimization)

top_products = df['Description'].value_counts().head(100).index
df_mba = df[df['Description'].isin(top_products)]

# Create transaction list
transactions = df_mba.groupby('InvoiceNo')['Description'].apply(list)

transactions.head()


#==========================================
# STEP 4 — ITEM SUPPORT CALCULATION
#==========================================
from collections import Counter

item_counts = Counter()

for items in transactions:
    item_counts.update(set(items))  # unique items per order

total_orders = len(transactions)

item_support = {
    item: count / total_orders
    for item, count in item_counts.items()
}


#==========================================
# STEP 5 — PAIR GENERATION (ASSOCIATIONS)
#==========================================
from itertools import combinations

pair_counts = Counter()

for items in transactions:
    pairs = combinations(set(items), 2)
    pair_counts.update(pairs)


#==========================================
# STEP 6 — CALCULATE SUPPORT, CONFIDENCE, LIFT
#==========================================
results = []

for (item1, item2), count in pair_counts.items():

    support_ab = count / total_orders
    support_a = item_support[item1]
    support_b = item_support[item2]

    confidence = support_ab / support_a
    lift = support_ab / (support_a * support_b)

    results.append([
        item1,
        item2,
        support_ab,
        confidence,
        lift
    ])

rules_df = pd.DataFrame(results, columns=[
    'Item_A',
    'Item_B',
    'Support',
    'Confidence',
    'Lift'
])

rules_df.head()


#==============================
# STEP 7 — FILTER STRONG RULES
#==============================
strong_rules = rules_df[
    (rules_df['Confidence'] > 0.3) &
    (rules_df['Lift'] > 1.5)
]


#==========================
# STEP 8 — SORT BEST RULES
#==========================
strong_rules = strong_rules.sort_values(by='Lift', ascending=False)

strong_rules.head(10)


#==========================================
# STEP 9 — RFM + CLV (FINAL UPGRADE)
#==========================================

# Create Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# RFM Aggregation
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': 'max',
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'LastPurchaseDate', 'Frequency', 'Monetary']

# Convert date
rfm['LastPurchaseDate'] = pd.to_datetime(rfm['LastPurchaseDate'])

# Recency
reference_date = rfm['LastPurchaseDate'].max()
rfm['Recency'] = (reference_date - rfm['LastPurchaseDate']).dt.days

# RFM Scores
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

# Segmentation
def segment(row):
    if int(row['R']) >= 4 and int(row['F']) >= 4:
        return "Champions"
    elif int(row['R']) <= 2 and int(row['F']) <= 2:
        return "Hibernating"
    elif int(row['R']) <= 2:
        return "At Risk"
    else:
        return "Loyal"

rfm['Segment'] = rfm.apply(segment, axis=1)


#====================
# STEP 10 — CLV
#====================

# Avg Order Value
rfm['AvgOrderValue'] = rfm['Monetary'] / rfm['Frequency']

# Lifespan assumption
rfm['CustomerLifespan'] = 12

# CLV Calculation
rfm['CLV'] = (
    rfm['AvgOrderValue'] *
    rfm['Frequency'] *
    rfm['CustomerLifespan']
)

rfm.head()


#====================
# STEP 11 — SAVE OUTPUTS
#====================
rfm.to_csv("rfm_output.csv", index=False)
strong_rules.to_csv("market_basket_rules.csv", index=False)

print("✅ Market Basket + RFM + CLV completed successfully!")

✅ Market Basket + RFM + CLV completed successfully!
